
# YOLO Colonies (Instance Segmentation)

Pipeline in this notebook:
1. Load `cropped_720` dataset.
2. Build a large offline-augmented pool (intentionally leaky bootstrap setup).
3. Split augmented pool into `train/val/test`.
4. Train `yolo26n-seg.pt` baseline.
5. Optionally train heavier models (`yolo26s-seg.pt`, `yolo26m-seg.pt`).


In [ ]:
from pathlib import Path
import json
import random
import shutil
import zlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Requested dataset path
DATASET_ROOT = Path("cropped_720")

# Fallback: auto-find cropped_720 folder inside repo if notebook is launched from project root.
if not DATASET_ROOT.exists():
    matches = sorted([p for p in Path(".").rglob("cropped_720") if p.is_dir()])
    if len(matches) == 1:
        DATASET_ROOT = matches[0]
    elif len(matches) > 1:
        DATASET_ROOT = matches[0]
        print(f"Found multiple cropped_720 folders, using: {DATASET_ROOT.resolve()}")

if not DATASET_ROOT.exists():
    raise FileNotFoundError("Could not find dataset root 'cropped_720'. Set DATASET_ROOT manually.")

SRC_IMG_DIR = DATASET_ROOT / "images" / "train"
SRC_LBL_DIR = DATASET_ROOT / "labels" / "train"

if not SRC_IMG_DIR.exists():
    raise FileNotFoundError(f"Source images dir not found: {SRC_IMG_DIR}")
if not SRC_LBL_DIR.exists():
    raise FileNotFoundError(f"Source labels dir not found: {SRC_LBL_DIR}")

WORK_ROOT = DATASET_ROOT.parent / "cropped_720_aug_leaky"
POOL_IMG_DIR = WORK_ROOT / "_pool" / "images"
POOL_LBL_DIR = WORK_ROOT / "_pool" / "labels"
SPLIT_ROOT = WORK_ROOT / "dataset"

print(f"DATASET_ROOT: {DATASET_ROOT.resolve()}")
print(f"WORK_ROOT: {WORK_ROOT.resolve()}")


In [ ]:

# Offline augmentation setup (label-safe transforms for YOLO-seg polygons)
OVERWRITE_WORK_ROOT = True

GEOM_MODES = ["none", "hflip", "vflip", "hvflip"]
PHOTO_MODES = ["none", "clahe", "gamma", "hsv", "blur", "noise"]
PHOTO_REPEATS = 3  # each repeat gets different deterministic random params

VARIANTS = []
for geom in GEOM_MODES:
    for photo in PHOTO_MODES:
        if photo == "none":
            name = "orig" if geom == "none" else f"{geom}_orig"
            VARIANTS.append({"name": name, "geom": geom, "photo": photo})
        else:
            for rep in range(1, PHOTO_REPEATS + 1):
                VARIANTS.append(
                    {
                        "name": f"{geom}_{photo}_r{rep}",
                        "geom": geom,
                        "photo": photo,
                    }
                )

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
OUTPUT_IMAGE_EXT = ".jpg"


def read_yolo_seg(label_path: Path):
    anns = []
    if not label_path.exists():
        return anns

    text = label_path.read_text(encoding="utf-8", errors="ignore")
    # Some labels contain literal "\\n" sequences instead of real newlines.
    text = text.replace("\\r", "\n").replace("\\n", "\n")

    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.replace(",", " ").split()
        if len(parts) < 7:
            continue

        cls = parts[0]
        coords = []
        for token in parts[1:]:
            try:
                coords.append(float(token))
            except ValueError:
                pass

        if len(coords) < 6:
            continue
        if len(coords) % 2 != 0:
            coords = coords[:-1]
        if len(coords) < 6:
            continue

        pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
        anns.append((cls, pts))
    return anns


def write_yolo_seg(label_path: Path, anns):
    label_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for cls, pts in anns:
        pts = np.clip(pts, 0.0, 1.0)
        flat = pts.reshape(-1)
        lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in flat))
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def transform_points(pts, geom):
    out = pts.copy()
    if geom in {"hflip", "hvflip"}:
        out[:, 0] = 1.0 - out[:, 0]
    if geom in {"vflip", "hvflip"}:
        out[:, 1] = 1.0 - out[:, 1]
    return np.clip(out, 0.0, 1.0)


def apply_geom(image, anns, geom):
    if geom == "none":
        return image, anns
    if geom == "hflip":
        out_img = cv2.flip(image, 1)
    elif geom == "vflip":
        out_img = cv2.flip(image, 0)
    elif geom == "hvflip":
        out_img = cv2.flip(image, -1)
    else:
        raise ValueError(f"Unknown geom transform: {geom}")

    out_anns = [(cls, transform_points(pts, geom)) for cls, pts in anns]
    return out_img, out_anns


def apply_photo(image, photo, rng):
    if photo == "none":
        return image

    img = image.copy()

    if photo == "clahe":
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clip = float(rng.uniform(2.0, 4.0))
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        return cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)

    if photo == "gamma":
        gamma = float(rng.uniform(0.7, 1.5))
        inv = 1.0 / gamma
        table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)], dtype=np.uint8)
        return cv2.LUT(img, table)

    if photo == "hsv":
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        h_gain = float(rng.uniform(-8, 8))
        s_gain = float(rng.uniform(0.75, 1.35))
        v_gain = float(rng.uniform(0.75, 1.35))
        hsv[..., 0] = (hsv[..., 0] + h_gain) % 180
        hsv[..., 1] = np.clip(hsv[..., 1] * s_gain, 0, 255)
        hsv[..., 2] = np.clip(hsv[..., 2] * v_gain, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    if photo == "blur":
        k = int(rng.choice([3, 5]))
        return cv2.GaussianBlur(img, (k, k), sigmaX=0)

    if photo == "noise":
        sigma = float(rng.uniform(5, 14))
        noise = rng.normal(0.0, sigma, size=img.shape).astype(np.float32)
        out = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        return out

    raise ValueError(f"Unknown photo transform: {photo}")


In [ ]:

# Build augmented pool first (intentional leak-prone bootstrap strategy)
if OVERWRITE_WORK_ROOT and WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)

POOL_IMG_DIR.mkdir(parents=True, exist_ok=True)
POOL_LBL_DIR.mkdir(parents=True, exist_ok=True)

src_images = sorted([p for p in SRC_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not src_images:
    raise RuntimeError(f"No source images found in: {SRC_IMG_DIR}")

written = 0
for i, img_path in enumerate(src_images, start=1):
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        continue

    in_label = SRC_LBL_DIR / f"{img_path.stem}.txt"
    anns = read_yolo_seg(in_label)

    for var in VARIANTS:
        out_stem = f"{img_path.stem}__{var['name']}"
        seed_local = SEED + zlib.crc32(out_stem.encode("utf-8"))
        rng = np.random.default_rng(seed_local)

        g_img, g_anns = apply_geom(img, anns, var["geom"])
        p_img = apply_photo(g_img, var["photo"], rng)

        out_img = POOL_IMG_DIR / f"{out_stem}{OUTPUT_IMAGE_EXT}"
        out_lbl = POOL_LBL_DIR / f"{out_stem}.txt"

        cv2.imwrite(str(out_img), p_img)
        write_yolo_seg(out_lbl, g_anns)
        written += 1

    if i % 10 == 0 or i == len(src_images):
        print(f"Augmented {i}/{len(src_images)} source images")

print(f"Source images: {len(src_images)}")
print(f"Variants per image: {len(VARIANTS)}")
print(f"Pool samples written: {written}")
print(f"Pool images dir: {POOL_IMG_DIR.resolve()}")


In [ ]:

# Split augmented pool into train/val/test AFTER augmentation (leaky by design)
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SPLIT_SEED = 42

if abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) > 1e-9:
    raise ValueError("Split ratios must sum to 1.0")

pool_images = sorted([p for p in POOL_IMG_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
pool_stems = [p.stem for p in pool_images]

rng = random.Random(SPLIT_SEED)
rng.shuffle(pool_stems)

n = len(pool_stems)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
n_test = n - n_train - n_val

splits = {
    "train": pool_stems[:n_train],
    "val": pool_stems[n_train:n_train + n_val],
    "test": pool_stems[n_train + n_val:],
}

for split_name in ["train", "val", "test"]:
    (SPLIT_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (SPLIT_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

for split_name, stems in splits.items():
    img_dst = SPLIT_ROOT / "images" / split_name
    lbl_dst = SPLIT_ROOT / "labels" / split_name

    for stem in stems:
        src_img = POOL_IMG_DIR / f"{stem}{OUTPUT_IMAGE_EXT}"
        src_lbl = POOL_LBL_DIR / f"{stem}.txt"

        shutil.copy2(src_img, img_dst / src_img.name)
        if src_lbl.exists():
            shutil.copy2(src_lbl, lbl_dst / src_lbl.name)
        else:
            (lbl_dst / f"{stem}.txt").write_text("", encoding="utf-8")

print(f"Total: {n}")
print(f"Train: {len(splits['train'])}")
print(f"Val:   {len(splits['val'])}")
print(f"Test:  {len(splits['test'])}")


In [ ]:

# Write data config for YOLO
DATA_YAML = SPLIT_ROOT / "data.yaml"

train_dir = (SPLIT_ROOT / "images" / "train").resolve().as_posix()
val_dir = (SPLIT_ROOT / "images" / "val").resolve().as_posix()
test_dir = (SPLIT_ROOT / "images" / "test").resolve().as_posix()

yaml_text = "\n".join([
    f'train: "{train_dir}"',
    f'val: "{val_dir}"',
    f'test: "{test_dir}"',
    "nc: 1",
    "names: [colony]",
]) + "\n"

DATA_YAML.write_text(yaml_text, encoding="utf-8")
print(DATA_YAML.resolve())
print(yaml_text)


In [ ]:

# Quick sanity checks

def count_objects(label_dir: Path):
    n_objects = 0
    n_files = 0
    for txt in sorted(label_dir.glob("*.txt")):
        n_files += 1
        for line in txt.read_text(encoding="utf-8").splitlines():
            if line.strip():
                n_objects += 1
    return n_files, n_objects

for split_name in ["train", "val", "test"]:
    img_dir = SPLIT_ROOT / "images" / split_name
    lbl_dir = SPLIT_ROOT / "labels" / split_name

    n_img = len(list(img_dir.glob(f"*{OUTPUT_IMAGE_EXT}")))
    n_lbl, n_obj = count_objects(lbl_dir)
    print(f"{split_name:5s} -> images={n_img}, labels={n_lbl}, objects={n_obj}")


In [ ]:
# Train baseline model: yolo26n-seg.pt (no online augmentations)
BASE_MODEL = "yolo26n-seg.pt"
PROJECT = "runs/colony_seg"
RUN_NAME = "yolo26n_seg_cropped720_leaky_aug"

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(DATA_YAML),
    task="segment",
    imgsz=720,
    epochs=50,
    batch=8,
    patience=80,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    # Disable online augmentations: use only offline-augmented dataset
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,
    flipud=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
)

run_dir = Path(results.save_dir)
print(f"Run dir: {run_dir.resolve()}")
print(f"Best weights: {(run_dir / 'weights' / 'best.pt').resolve()}")


In [ ]:
# Plot training curves (from results.csv if available, else from saved PNGs)
def resolve_train_run_dir():
    candidates = []

    def has_train_artifacts(d: Path):
        if not d.exists() or not d.is_dir():
            return False
        return (
            (d / "results.csv").exists()
            or (d / "results.png").exists()
            or any(d.glob("*curve*.png"))
        )

    if "run_dir" in globals():
        try:
            d = Path(run_dir)
            if has_train_artifacts(d):
                candidates.append(d)
        except Exception:
            pass

    run_name = globals().get("RUN_NAME", "yolo26n_seg_cropped720_leaky_aug")
    explicit = Path("runs") / "segment" / "runs" / "colony_seg" / run_name
    if has_train_artifacts(explicit):
        candidates.append(explicit)

    # also check absolute project path if notebook cwd is different
    explicit_abs = Path("C:/ColonyNet/runs/segment/runs/colony_seg") / run_name
    if has_train_artifacts(explicit_abs):
        candidates.append(explicit_abs)

    for base in [Path("runs") / "segment" / "runs" / "colony_seg", Path("C:/ColonyNet/runs/segment/runs/colony_seg")]:
        if base.exists():
            for d in base.iterdir():
                if d.is_dir() and has_train_artifacts(d) and not d.name.endswith("_test"):
                    candidates.append(d)

    if not candidates:
        raise FileNotFoundError("No train run directory with plots/results found.")

    candidates = sorted(set(candidates), key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]

run_dir = resolve_train_run_dir()
print(f"Using run_dir: {run_dir}")

results_csv = run_dir / "results.csv"
if results_csv.exists():
    train_df = pd.read_csv(results_csv)
    if "epoch" not in train_df.columns:
        train_df["epoch"] = np.arange(len(train_df))

    loss_cols = [c for c in train_df.columns if "loss" in c.lower() and "lr" not in c.lower()]
    metric_cols = [
        c for c in train_df.columns
        if any(k in c.lower() for k in ["precision", "recall", "map50", "map", "fitness"])
        and "class" not in c.lower()
    ]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    if loss_cols:
        for col in loss_cols:
            axes[0].plot(train_df["epoch"], train_df[col], label=col)
        axes[0].legend(loc="best", fontsize=8)
    else:
        axes[0].text(0.5, 0.5, "No loss columns found", ha="center", va="center")
    axes[0].set_title("Loss curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    if metric_cols:
        for col in metric_cols:
            axes[1].plot(train_df["epoch"], train_df[col], label=col)
        axes[1].legend(loc="best", fontsize=8)
    else:
        axes[1].text(0.5, 0.5, "No metric columns found", ha="center", va="center")
    axes[1].set_title("Metrics curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Metric")

    plt.tight_layout()
    plt.show()
else:
    print(f"results.csv not found in {run_dir}; showing saved PNG curves instead.")

# Show Ultralytics rendered summary image if present
results_png = run_dir / "results.png"
if results_png.exists():
    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(results_png))
    plt.title(results_png.name)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Show train-time Box/Mask curve images if present
curve_candidates = [
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "MaskPR_curve.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "MaskF1_curve.png",
]
curve_paths = [run_dir / n for n in curve_candidates if (run_dir / n).exists()]

if curve_paths:
    ncols = 2
    nrows = int(np.ceil(len(curve_paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, p in zip(axes, curve_paths):
        ax.imshow(Image.open(p))
        ax.set_title(p.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Evaluate best checkpoint on test split + save summary metrics
best_weights = run_dir / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"best.pt not found: {best_weights}")

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=720,
    project=PROJECT,
    name=f"{RUN_NAME}_test",
    exist_ok=True,
    plots=True,
    save_json=True,
)

def to_float(v):
    try:
        return float(v)
    except Exception:
        return float("nan")

def summarize_section(section, prefix):
    if section is None:
        return {
            f"precision_{prefix}": float("nan"),
            f"recall_{prefix}": float("nan"),
            f"mAP50_{prefix}": float("nan"),
            f"mAP50_95_{prefix}": float("nan"),
        }
    return {
        f"precision_{prefix}": to_float(getattr(section, "mp", float("nan"))),
        f"recall_{prefix}": to_float(getattr(section, "mr", float("nan"))),
        f"mAP50_{prefix}": to_float(getattr(section, "map50", float("nan"))),
        f"mAP50_95_{prefix}": to_float(getattr(section, "map", float("nan"))),
    }

metrics_summary = {}
metrics_summary.update(summarize_section(getattr(test_metrics, "box", None), "B"))
metrics_summary.update(summarize_section(getattr(test_metrics, "seg", None), "M"))
metrics_summary["fitness"] = to_float(getattr(test_metrics, "fitness", float("nan")))

test_dir = Path(test_metrics.save_dir)
metrics_df = pd.DataFrame({"value": metrics_summary})
try:
    display(metrics_df)
except Exception:
    print(metrics_df)

metrics_json_path = test_dir / "test_metrics_summary.json"
with metrics_json_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

print(f"Test run dir: {test_dir}")
print(f"Saved metrics JSON: {metrics_json_path}")


In [ ]:
# Show evaluation plots (PR/F1/Confusion Matrix) if present (auto-resolve test_dir)
def resolve_test_run_dir():
    candidates = []

    if "test_dir" in globals():
        try:
            d = Path(test_dir)
            if d.exists():
                candidates.append(d)
        except Exception:
            pass

    run_name = globals().get("RUN_NAME", "yolo26n_seg_cropped720_leaky_aug")
    explicit = Path("runs") / "segment" / "runs" / "colony_seg" / f"{run_name}_test"
    if explicit.exists():
        candidates.append(explicit)

    base = Path("runs") / "segment" / "runs" / "colony_seg"
    if base.exists():
        for d in base.iterdir():
            if d.is_dir() and d.name.endswith("_test"):
                candidates.append(d)

    if not candidates:
        raise FileNotFoundError("No *_test run directory found.")

    candidates = sorted(set(candidates), key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]

test_dir = resolve_test_run_dir()
print(f"Using test_dir: {test_dir}")

plot_candidates = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "MaskPR_curve.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "MaskF1_curve.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]

plot_paths = [test_dir / p for p in plot_candidates if (test_dir / p).exists()]

# Fallback: include any other curve/confusion images generated by Ultralytics
for pat in ["*curve*.png", "*confusion*.png"]:
    for p in sorted(test_dir.glob(pat)):
        if p not in plot_paths:
            plot_paths.append(p)

if not plot_paths:
    print("No evaluation plot images found in test run directory.")
else:
    print(f"Found {len(plot_paths)} plot images")
    ncols = 2
    nrows = int(np.ceil(len(plot_paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, p in zip(axes, plot_paths):
        img = Image.open(p)
        ax.imshow(img)
        ax.set_title(p.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# Preview segmentation masks and contours only (no bounding boxes)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
NUM_PREVIEW = 6
PRED_CONF = 0.25
PRED_IOU = 0.6
MASK_ALPHA = 0.35


def resolve_best_weights_for_preview():
    candidates = []

    if "best_weights" in globals():
        try:
            p = Path(best_weights)
            if p.exists():
                candidates.append(p)
        except Exception:
            pass

    if "run_dir" in globals():
        try:
            p = Path(run_dir) / "weights" / "best.pt"
            if p.exists():
                candidates.append(p)
        except Exception:
            pass

    run_name = globals().get("RUN_NAME", "yolo26n_seg_cropped720_leaky_aug")
    for base in [
        Path("runs") / "segment" / "runs" / "colony_seg",
        Path("C:/ColonyNet/runs/segment/runs/colony_seg"),
    ]:
        p = base / run_name / "weights" / "best.pt"
        if p.exists():
            candidates.append(p)

    for base in [
        Path("runs") / "segment" / "runs" / "colony_seg",
        Path("C:/ColonyNet/runs/segment/runs/colony_seg"),
    ]:
        if base.exists():
            for d in base.iterdir():
                p = d / "weights" / "best.pt"
                if d.is_dir() and p.exists() and not d.name.endswith("_test"):
                    candidates.append(p)

    if not candidates:
        raise FileNotFoundError("Could not find best.pt for preview.")

    candidates = sorted(set(candidates), key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


def resolve_preview_image_dir():
    dirs = []

    if "SPLIT_ROOT" in globals():
        for split in ["test", "val", "train"]:
            d = Path(SPLIT_ROOT) / "images" / split
            if d.exists():
                dirs.append(d)

    for d in [
        Path("cropped_720_aug_leaky/dataset/images/test"),
        Path("cropped_720_aug_leaky/dataset/images/val"),
        Path("cropped_720_aug_leaky/dataset/images/train"),
        Path("C:/ColonyNet/cropped_720_aug_leaky/dataset/images/test"),
        Path("C:/ColonyNet/cropped_720_aug_leaky/dataset/images/val"),
        Path("C:/ColonyNet/cropped_720_aug_leaky/dataset/images/train"),
    ]:
        if d.exists():
            dirs.append(d)

    if not dirs:
        raise FileNotFoundError("Could not find preview image directory.")

    dirs = sorted(set(dirs), key=lambda x: x.stat().st_mtime, reverse=True)
    return dirs[0]


def color_for_idx(i):
    # deterministic distinct colors (RGB)
    palette = [
        (255, 80, 80), (80, 200, 255), (120, 255, 120), (255, 200, 80),
        (210, 120, 255), (255, 120, 180), (120, 255, 220), (255, 255, 120),
    ]
    return np.array(palette[i % len(palette)], dtype=np.float32)


weights_path = resolve_best_weights_for_preview()
preview_dir = resolve_preview_image_dir()

preview_images = sorted([p for p in preview_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])[:NUM_PREVIEW]
if not preview_images:
    raise RuntimeError(f"No images found in {preview_dir}")

print(f"Using weights: {weights_path}")
print(f"Using images from: {preview_dir}")

preview_model = YOLO(str(weights_path))
preview_results = preview_model.predict(
    source=[str(p) for p in preview_images],
    conf=PRED_CONF,
    iou=PRED_IOU,
    save=False,
    verbose=False,
)

n = len(preview_images)
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for ax, img_path, res in zip(axes, preview_images, preview_results):
    bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if bgr is None:
        ax.set_title(f"{img_path.name} (read error)")
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    overlay = rgb.copy()

    n_masks = 0
    if res.masks is not None and getattr(res.masks, "xy", None) is not None:
        for k, poly in enumerate(res.masks.xy):
            if poly is None or len(poly) < 3:
                continue

            pts = np.round(poly).astype(np.int32)
            color = color_for_idx(k)

            mask = np.zeros(overlay.shape[:2], dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)

            for ch in range(3):
                overlay[..., ch] = np.where(
                    mask > 0,
                    overlay[..., ch] * (1.0 - MASK_ALPHA) + color[ch] * MASK_ALPHA,
                    overlay[..., ch],
                )

            cv2.polylines(overlay, [pts], True, color.tolist(), 2, lineType=cv2.LINE_AA)
            n_masks += 1

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    ax.imshow(out)
    ax.set_title(f"{img_path.name} | masks: {n_masks}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Optional: train heavier models after baseline
HEAVIER_MODELS = [
    "yolo26s-seg.pt",
    "yolo26m-seg.pt",
]
RUN_HEAVIER = False

if RUN_HEAVIER:
    for ckpt in HEAVIER_MODELS:
        run_name = f"{Path(ckpt).stem}_cropped720_leaky_aug"
        print(f"\nTraining {ckpt} -> {run_name}")

        model = YOLO(ckpt)
        model.train(
            data=str(DATA_YAML),
            task="segment",
            imgsz=720,
            epochs=200,
            batch=8,
            patience=80,
            project=PROJECT,
            name=run_name,
            exist_ok=True,
            plots=True,
            # Disable online augmentations: use only offline-augmented dataset
            degrees=0.0,
            translate=0.0,
            scale=0.0,
            shear=0.0,
            perspective=0.0,
            fliplr=0.0,
            flipud=0.0,
            hsv_h=0.0,
            hsv_s=0.0,
            hsv_v=0.0,
            mosaic=0.0,
            mixup=0.0,
            copy_paste=0.0,
            erasing=0.0,
        )
else:
    print("Set RUN_HEAVIER = True to train yolo26s-seg.pt / yolo26m-seg.pt")
